# **Proyecto Final – Optimización e Inteligencia Artificial**
## Predicción del riesgo de incumplimiento crediticio (Home Credit Default Risk): comparación de optimizadores y optimización de hiperparámetros

**Integrantes:**
- Kevin Leandro Ramos Luna
- Daniel Felipe Garzón Acosta
- Ángel Efraín Pimienta Durán
- Daniel Ortega
- Manuel Mera Mera

**Docente:** María C. Torres

**Base de datos:** [Home Credit Default Risk – Kaggle](https://www.kaggle.com/competitions/home-credit-default-risk)

**Estructura del notebook:**

- **Parte 0.** Configuración del entorno
- **Parte 1.** Datos: exploración y preprocesamiento
- **Parte 2.** Modelos de referencia (regresión logística y MLP base)
- **Parte 3.** Comparación de optimizadores para el entrenamiento del MLP
- **Parte 4.** Optimización de hiperparámetros del MLP (Random Search, Bayesiana, Hyperband)
- **Parte 5.** LightGBM + Optuna (TPE vs búsqueda aleatoria)
- **Parte 6.** Evaluación final y comparación global

## PARTE 0: Configuración del entorno

El notebook se ejecuta en **Google Colab con GPU** (*Entorno de ejecución → Cambiar tipo de entorno → GPU*).
Los datos se leen desde Google Drive: subir **`application_train.csv.zip`** (35 MB, recomendado) o `application_train.csv` (166 MB) a la carpeta `MyDrive/home-credit-default-risk/`. Si está en otra carpeta de Drive, el notebook lo busca automáticamente. *pandas* lee el `.zip` directamente.

In [ ]:
!pip install -q keras-tuner optuna lightgbm

In [ ]:
#Importacion de librerias
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn import metrics

#Avisos inofensivos: nombres de columnas en LightGBM y convergencia del proceso gaussiano de Keras Tuner
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="The optimal value found for dimension")

In [ ]:
#TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras

#Semilla para que los resultados sean reproducibles
SEED = 42
keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
#Rutas: en Colab se monta Google Drive, en local se usa la carpeta del proyecto
import glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    #Se busca application_train.csv.zip (recomendado: 35 MB) o application_train.csv (166 MB)
    #en todo Drive y en /content (si se subio directo a la sesion)
    encontrados = []
    for nombre in ["application_train.csv.zip", "application_train.csv"]:
        encontrados += glob.glob(f"/content/drive/MyDrive/**/{nombre}", recursive=True) + glob.glob(f"/content/{nombre}")
    if not encontrados:
        raise FileNotFoundError("No se encontró application_train.csv.zip ni application_train.csv. Súbalo a Google Drive, "
                                "idealmente en MyDrive/home-credit-default-risk/, y vuelva a ejecutar esta celda.")
    for ruta in encontrados:
        print(f"Encontrado: {ruta} ({os.path.getsize(ruta) / 1e6:.1f} MB)")
    #Se prefiere el .zip: si su subida quedo incompleta, pandas da error en vez de leer datos parciales
    DATA_FILE = encontrados[0]
    FIG_DIR = "/content/drive/MyDrive/OptimizacionEIA_Proyecto/figuras"
else:
    DATA_FILE = "../ProyectoFInal/home-credit-default-risk/application_train.csv"
    FIG_DIR = "figuras"
os.makedirs(FIG_DIR, exist_ok=True)
print("Datos:", DATA_FILE)

def guardar_fig(nombre):
    #Guarda la figura actual en alta resolucion para usarla en el poster
    plt.savefig(f"{FIG_DIR}/{nombre}.png", dpi=300, bbox_inches="tight")

## PARTE 1: Datos – exploración y preprocesamiento

Se usa la tabla principal `application_train.csv`: cada fila es una solicitud de crédito y la variable objetivo **TARGET** indica si el cliente tuvo dificultades de pago (**1**) o no (**0**). Es un problema de **clasificación binaria**.

In [ ]:
#Carga de datos
df = pd.read_csv(DATA_FILE)
print("Dimensiones:", df.shape)
#El archivo completo tiene 307.511 registros (166 MB); si tiene menos, la subida a Drive quedo incompleta
assert len(df) == 307511, (f"El archivo tiene {len(df):,} registros en vez de 307.511: está incompleto. "
                           "Suba application_train.csv.zip a Drive y espere a que termine la subida.")
print("Variables numéricas:", df.select_dtypes(include="number").shape[1])
print("Variables categóricas:", df.select_dtypes(exclude="number").shape[1])
print("Datos faltantes (total):", df.isna().sum().sum())
df.head()

### Exploración de los datos

In [ ]:
#Distribucion de la variable objetivo
conteo = df["TARGET"].value_counts()
print(conteo)
print("Proporción de incumplimiento: %.2f %%" % (100 * df["TARGET"].mean()))

plt.figure(figsize=(5, 4))
ax = sns.barplot(x=["Paga (0)", "Incumple (1)"], y=conteo.values, hue=["Paga (0)", "Incumple (1)"],
                 palette=["tab:blue", "tab:red"], legend=False)
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v:,}\n({100 * v / conteo.sum():.1f} %)", ha="center", va="bottom")
plt.title("Distribución de la variable objetivo")
plt.ylabel("Número de clientes")
plt.ylim(0, conteo.max() * 1.15)
guardar_fig("01_desbalance_target")
plt.show()

El conjunto está **muy desbalanceado** (≈ 8 % de incumplimiento). Un modelo que siempre prediga "paga" tendría ≈ 92 % de *accuracy*, por eso usamos **ROC-AUC** como métrica principal (es también la métrica de la competencia en Kaggle), complementada con PR-AUC, *recall* y F1.

In [ ]:
#Porcentaje de datos faltantes por columna
faltantes = (df.isna().mean() * 100).sort_values(ascending=False)
print("Columnas con faltantes:", (faltantes > 0).sum())
print("Columnas con más del 60 % de faltantes:", (faltantes > 60).sum())
faltantes.head(15)

In [ ]:
#Variables numericas mas correlacionadas con TARGET
corr = df.select_dtypes(include="number").corr()["TARGET"].drop(["TARGET", "SK_ID_CURR"])
top_corr = corr.reindex(corr.abs().sort_values(ascending=False).index).head(10)
print(top_corr)

plt.figure(figsize=(7, 4))
top_corr.sort_values().plot(kind="barh", color=["tab:red" if v > 0 else "tab:blue" for v in top_corr.sort_values()])
plt.title("Top 10 correlaciones con TARGET")
plt.grid(True, axis="x")
guardar_fig("02_correlaciones_target")
plt.show()

In [ ]:
#Distribucion de las variables mas relevantes segun TARGET
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"]):
    datos = df[col] / -365 if col == "DAYS_BIRTH" else df[col]
    sns.kdeplot(datos[df["TARGET"] == 0], ax=ax, label="Paga (0)", fill=True)
    sns.kdeplot(datos[df["TARGET"] == 1], ax=ax, label="Incumple (1)", fill=True, color="tab:red")
    ax.set_title("Edad (años)" if col == "DAYS_BIRTH" else col)
    ax.legend()
guardar_fig("03_kde_variables_clave")
plt.show()

### Preprocesamiento

1. **Anomalía en `DAYS_EMPLOYED`:** el valor 365243 (≈ 1000 años) corresponde a pensionados/desempleados → se reemplaza por NaN y se crea la bandera `DAYS_EMPLOYED_ANOM`.
2. **`CODE_GENDER = XNA`:** solo 4 registros, se eliminan.
3. **Ingeniería de variables:** razones financieras (crédito/ingreso, anualidad/ingreso, plazo del crédito, proporción de vida laboral) y combinaciones de las `EXT_SOURCE`.
4. **Columnas con más del 60 % de faltantes** se eliminan.
5. **Codificación:** variables categóricas binarias → 0/1; el resto → *one-hot* (los faltantes quedan como una categoría más).
6. **Partición estratificada** 70 % entrenamiento / 15 % validación / 15 % prueba.
7. **Imputación** (mediana) y **estandarización**, ajustadas **solo con el conjunto de entrenamiento** para no filtrar información.

In [ ]:
datos = df.copy()

#1. Anomalia de DAYS_EMPLOYED
datos["DAYS_EMPLOYED_ANOM"] = (datos["DAYS_EMPLOYED"] == 365243).astype(int)
datos["DAYS_EMPLOYED"] = datos["DAYS_EMPLOYED"].replace(365243, np.nan)

#2. Registros con genero desconocido
datos = datos[datos["CODE_GENDER"] != "XNA"]

#3. Ingenieria de variables
datos["CREDIT_INCOME_RATIO"] = datos["AMT_CREDIT"] / datos["AMT_INCOME_TOTAL"]
datos["ANNUITY_INCOME_RATIO"] = datos["AMT_ANNUITY"] / datos["AMT_INCOME_TOTAL"]
datos["CREDIT_TERM"] = datos["AMT_ANNUITY"] / datos["AMT_CREDIT"]
datos["DAYS_EMPLOYED_RATIO"] = datos["DAYS_EMPLOYED"] / datos["DAYS_BIRTH"]
datos["EXT_SOURCE_MEAN"] = datos[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
datos["EXT_SOURCE_PROD"] = datos["EXT_SOURCE_1"] * datos["EXT_SOURCE_2"] * datos["EXT_SOURCE_3"]

#4. Columnas con mas del 60 % de faltantes
cols_drop = datos.columns[datos.isna().mean() > 0.60]
datos = datos.drop(columns=cols_drop)
print("Columnas eliminadas por faltantes:", len(cols_drop))
print("Dimensiones:", datos.shape)

In [ ]:
#5. Codificacion de variables categoricas
cat_cols = datos.select_dtypes(exclude="number").columns
binarias = [c for c in cat_cols if datos[c].nunique(dropna=False) == 2]
multiclase = [c for c in cat_cols if c not in binarias]

for col in binarias:
    datos[col] = pd.factorize(datos[col])[0]
datos = pd.get_dummies(datos, columns=multiclase, dummy_na=True)
print("Binarias:", binarias)
print("Categóricas con one-hot:", len(multiclase))

#Caracteristicas y variable objetivo
X = datos.drop(columns=["SK_ID_CURR", "TARGET"]).astype("float32")
y = datos["TARGET"].values.astype("float32")
feature_names = X.columns.tolist()
print("X:", X.shape, " y:", y.shape)

In [ ]:
#6. Particion estratificada: 70 % entrenamiento, 15 % validacion, 15 % prueba
x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
x_valid, x_test, y_valid, y_test = train_test_split(x_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

#7. Imputacion (mediana) y estandarizacion ajustadas solo con entrenamiento
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
x_train = scaler.fit_transform(imputer.fit_transform(x_train)).astype("float32")
x_valid = scaler.transform(imputer.transform(x_valid)).astype("float32")
x_test = scaler.transform(imputer.transform(x_test)).astype("float32")

for nombre, xs, ys in [("entrenamiento", x_train, y_train), ("validación", x_valid, y_valid), ("prueba", x_test, y_test)]:
    print(f"Datos de {nombre}: {xs.shape}  - incumplimiento: {100 * ys.mean():.2f} %")

In [ ]:
#Pesos por clase para compensar el desbalance (se usan en todos los modelos)
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(pesos[0]), 1: float(pesos[1])}
print("Pesos por clase:", class_weight)

### Funciones auxiliares de evaluación

Basadas en las de los talleres, pero adaptadas a **clasificación desbalanceada**:

- El **umbral de decisión** se elige en **validación** (el que maximiza F1) y luego se aplica en prueba.
- La **velocidad de convergencia** se mide como el número de épocas necesarias para alcanzar un AUC de validación ≥ `UMBRAL_AUC` (se calcula desde el historial, así que el umbral se puede cambiar sin re-entrenar).
- Se registra la **época con el mejor AUC de validación** y cuánto cae el AUC desde ese punto hasta la última época: una caída grande indica **sobreajuste**.

In [ ]:
resultados = {}
UMBRAL_AUC = 0.74   #AUC de validacion "aceptable" para medir velocidad de convergencia

def predecir_proba(model, x):
    #Probabilidad de la clase 1 para modelos de sklearn/LightGBM o de Keras
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]
    return model.predict(x, batch_size=4096, verbose=0).ravel()

def evaluar(nombre, model, history=None, tiempo=np.nan):
    p_valid = predecir_proba(model, x_valid)
    p_test = predecir_proba(model, x_test)
    #Umbral que maximiza F1 en validacion
    prec, rec, umbrales = metrics.precision_recall_curve(y_valid, p_valid)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    umbral = umbrales[np.argmax(f1[:-1])]
    y_pred = (p_test >= umbral).astype(int)

    fila = {"Tiempo (s)": round(tiempo, 2)}
    if history is not None:
        val_auc = np.array(history.history["val_auc"])
        fila["Época mejor val AUC"] = int(np.argmax(val_auc)) + 1
        fila["Caída val AUC (mejor − final)"] = val_auc.max() - val_auc[-1]
    fila.update({
        "Val AUC": metrics.roc_auc_score(y_valid, p_valid),
        "Test AUC": metrics.roc_auc_score(y_test, p_test),
        "Test PR-AUC": metrics.average_precision_score(y_test, p_test),
        "Umbral": umbral,
        "Test Recall": metrics.recall_score(y_test, y_pred),
        "Test Precision": metrics.precision_score(y_test, y_pred),
        "Test F1": metrics.f1_score(y_test, y_pred),
    })
    resultados[nombre] = fila
    print(nombre)
    for k in ["Tiempo (s)", "Val AUC", "Test AUC", "Test PR-AUC", "Test F1"]:
        print(f"  {k}: {fila[k]:.4f}")

def graficar(history, titulo):
    #Curvas de aprendizaje: perdida y AUC
    h = pd.DataFrame(history.history)
    h.index = h.index + 1
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    h[["loss", "val_loss"]].plot(ax=ax[0], title="Pérdida (entropía cruzada)")
    h[["auc", "val_auc"]].plot(ax=ax[1], title="ROC-AUC")
    for a in ax:
        a.grid(True)
        a.set_xlabel("Época")
    fig.suptitle(titulo)
    plt.show()

## PARTE 2: Modelos de referencia

Antes de optimizar se construyen dos modelos de referencia:

- **Regresión logística** (`class_weight="balanced"`): modelo lineal clásico en *credit scoring*.
- **MLP base** (Keras): será el "modelo sin optimizar" contra el cual se medirá la mejora de las Partes 3 y 4.

### Regresión logística

In [ ]:
inicio = time.time()
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, C=0.1)
log_reg.fit(x_train, y_train)
tiempo = time.time() - inicio
evaluar("Regresión logística", log_reg, tiempo=tiempo)

### MLP base

Hiperparámetros **fijos** (se mantienen en la comparación de optimizadores de la Parte 3):

- **Capas ocultas:** 3 (256, 128 y 64 neuronas), activación ReLU, *dropout* de 0.3
- **Capa de salida:** 1 neurona con activación sigmoide
- **Función de pérdida:** entropía cruzada binaria
- **Optimizador:** Adam, tasa de aprendizaje 0.001
- **Épocas:** 30 · **Batch size:** 512 (el conjunto de entrenamiento tiene ≈ 215.000 registros)
- **Pesos por clase** para compensar el desbalance

**Detalles de evaluación:**

- **Mejor época:** todos los modelos entrenan las 30 épocas completas (mismo presupuesto, curvas completas), pero al final se **restauran los pesos de la época con mejor AUC de validación**, igual que haría un *early stopping*. Así el AUC de prueba no se castiga por el sobreajuste de las últimas épocas.
- **Precisión de la métrica:** Keras calcula el AUC con 200 umbrales fijos por defecto, y eso **subestima** el AUC cuando las probabilidades se concentran cerca de 0 o 1. Se usan 1000 umbrales para que el AUC por época se parezca al exacto de *scikit-learn*.

In [ ]:
#Parametros fijos del MLP base
LR = 0.001
EPOCHS = 30
BATCH_SIZE = 512
N_UMBRALES = 1000   #umbrales para el AUC de Keras (por defecto 200 subestima el AUC)

class MejoresPesos(keras.callbacks.Callback):
    #Al terminar el entrenamiento restaura los pesos de la epoca con mejor val_auc
    def on_train_begin(self, logs=None):
        self.mejor, self.pesos = -np.inf, None
    def on_epoch_end(self, epoch, logs=None):
        if logs["val_auc"] > self.mejor:
            self.mejor, self.pesos = logs["val_auc"], self.model.get_weights()
    def on_train_end(self, logs=None):
        if self.pesos is not None:
            self.model.set_weights(self.pesos)

def construir_mlp(optimizador, capas=(256, 128, 64), dropout=0.3, seed=SEED):
    #Construye y compila el MLP con el optimizador dado
    keras.utils.set_random_seed(seed)
    model = keras.models.Sequential([keras.layers.Input(shape=x_train.shape[1:])])
    for n in capas:
        model.add(keras.layers.Dense(n, activation="relu"))
        model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(loss="binary_crossentropy",
                  optimizer=optimizador,
                  metrics=[keras.metrics.AUC(name="auc", num_thresholds=N_UMBRALES),
                           keras.metrics.AUC(curve="PR", name="pr_auc", num_thresholds=N_UMBRALES)])
    return model

def entrenar(model, epochs=EPOCHS, callbacks=None, verbose=2):
    #Entrena el modelo y devuelve el historial y el tiempo de entrenamiento
    inicio = time.time()
    history = model.fit(x_train, y_train, epochs=epochs, batch_size=BATCH_SIZE,
                        validation_data=(x_valid, y_valid), class_weight=class_weight,
                        callbacks=(callbacks or []) + [MejoresPesos()], verbose=verbose)
    return history, time.time() - inicio

In [ ]:
model_base = construir_mlp(keras.optimizers.Adam(learning_rate=LR))
model_base.summary()
history_base, tiempo = entrenar(model_base)
evaluar("MLP base (Adam)", model_base, history_base, tiempo)
graficar(history_base, "Curva de aprendizaje - MLP base")

In [ ]:
tabla_base = pd.DataFrame(resultados).T
tabla_base

### Conclusiones Parte 2

- **Desempeño:** la regresión logística (AUC de prueba **0.7497**) y el MLP base (AUC de prueba **0.7494**) tienen prácticamente el mismo desempeño, también en PR-AUC (0.2331 vs 0.2315) y F1 (0.2985 vs 0.2960). Con hiperparámetros fijos, la red neuronal no aporta capacidad predictiva adicional frente al modelo lineal, y tarda ≈ 5 veces más en entrenar (69 s vs 13 s).
- **Sobreajuste:** las curvas del MLP muestran sobreajuste claro:
  - El AUC de entrenamiento sube de forma continua hasta **0.808** en la época 30.
  - El AUC de validación alcanza su máximo (**0.7487**) en la **época 6** y luego cae hasta 0.730 (caída de 0.018).
  - Sin restaurar la mejor época, el modelo evaluado sería claramente peor. Esto motiva comparar optimizadores (Parte 3) y optimizar la regularización y la arquitectura (Parte 4).

## PARTE 3: Comparación de optimizadores para el entrenamiento del MLP

Misma arquitectura y mismos hiperparámetros que el MLP base (tasa de aprendizaje 0.001, 30 épocas, batch 512). Solo cambia el optimizador:

| Vistos en clase | De la literatura |
|---|---|
| SGD, Momentum, Nesterov, RMSprop, Adam | Adagrad, Nadam, AdamW |

Cada optimizador se entrena con **3 semillas** y se reporta la media ± la desviación estándar.

**Métricas:**

- AUC de validación por época
- Épocas hasta AUC ≥ `UMBRAL_AUC` (velocidad de convergencia)
- Época del mejor AUC y caída posterior (sobreajuste)
- Tiempo de entrenamiento
- AUC de prueba con los pesos de la mejor época

In [ ]:
#Funcion para crear una instancia nueva de cada optimizador
def crear_optimizador(nombre, lr=LR):
    return {"SGD":      lambda: keras.optimizers.SGD(learning_rate=lr),
            "Momentum": lambda: keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
            "Nesterov": lambda: keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True),
            "Adagrad":  lambda: keras.optimizers.Adagrad(learning_rate=lr),
            "RMSprop":  lambda: keras.optimizers.RMSprop(learning_rate=lr),
            "Adam":     lambda: keras.optimizers.Adam(learning_rate=lr),
            "Nadam":    lambda: keras.optimizers.Nadam(learning_rate=lr),
            "AdamW":    lambda: keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4)}[nombre]()

OPTIMIZADORES = ["SGD", "Momentum", "Nesterov", "Adagrad", "RMSprop", "Adam", "Nadam", "AdamW"]
SEMILLAS = [42, 7, 123]

### Entrenamiento de los 24 modelos (8 optimizadores × 3 semillas)

Cada entrenamiento se guarda en un archivo JSON en Drive apenas termina. Si Colab se desconecta, al volver a ejecutar la celda **solo se entrenan los modelos que faltan**.

In [ ]:
#Archivo donde se guardan las corridas (metricas + historial por epoca)
RES_DIR = os.path.join(os.path.dirname(FIG_DIR), "resultados")
os.makedirs(RES_DIR, exist_ok=True)
ARCHIVO_OPT = f"{RES_DIR}/parte3_optimizadores_v2.json"   #v2: mejor epoca + AUC con 1000 umbrales
corridas = {}
if os.path.exists(ARCHIVO_OPT):
    with open(ARCHIVO_OPT) as f:
        corridas = json.load(f)
print("Corridas ya guardadas:", len(corridas), "de", len(OPTIMIZADORES) * len(SEMILLAS))

#Se guardan los resultados de la parte 2 y se reinicia el diccionario
resultados_base = dict(resultados)
resultados = {}

In [ ]:
for nombre in OPTIMIZADORES:
    for semilla in SEMILLAS:
        clave = f"{nombre} (seed {semilla})"
        if clave in corridas:
            print(clave, "-> ya entrenado, se omite")
            continue
        keras.backend.clear_session()
        model = construir_mlp(crear_optimizador(nombre), seed=semilla)
        history, tiempo = entrenar(model, verbose=0)
        evaluar(clave, model, history, tiempo)
        corridas[clave] = {"Optimizador": nombre, "Semilla": semilla,
                           **{k: float(v) for k, v in resultados[clave].items()},
                           "historia": {k: [float(x) for x in v] for k, v in history.history.items()}}
        with open(ARCHIVO_OPT, "w") as f:
            json.dump(corridas, f)

### Comparación de optimizadores

In [ ]:
#Tabla por corrida y resumen por optimizador (media y desviacion de las 3 semillas)
COL_EPOCAS = f"Épocas hasta val AUC≥{UMBRAL_AUC}"
COLS = ["Tiempo (s)", COL_EPOCAS, "Época mejor val AUC", "Caída val AUC (mejor − final)",
        "Val AUC", "Test AUC", "Test PR-AUC", "Test F1"]

def epocas_hasta_umbral(val_auc, umbral=UMBRAL_AUC):
    #Primera epoca en la que el AUC de validacion alcanza el umbral (NaN si nunca lo alcanza)
    alcanzo = np.where(np.array(val_auc) >= umbral)[0]
    return alcanzo[0] + 1 if len(alcanzo) else np.nan

tabla_corridas = pd.DataFrame([{**{k: v for k, v in c.items() if k != "historia"},
                                COL_EPOCAS: epocas_hasta_umbral(c["historia"]["val_auc"])}
                               for c in corridas.values()])
resumen_opt = tabla_corridas.groupby("Optimizador")[COLS].agg(["mean", "std"]).reindex(OPTIMIZADORES)
alcanzan = tabla_corridas.groupby("Optimizador")[COL_EPOCAS].count().reindex(OPTIMIZADORES)

#Tabla legible "media ± desviacion"
formatos = {"Tiempo (s)": "{:.1f}", COL_EPOCAS: "{:.1f}", "Época mejor val AUC": "{:.1f}"}
tabla_opt = pd.DataFrame({
    c: resumen_opt[c]["mean"].map(formatos.get(c, "{:.4f}").format) + " ± "
       + resumen_opt[c]["std"].map(formatos.get(c, "{:.4f}").format)
    for c in COLS})
tabla_opt[COL_EPOCAS] = tabla_opt[COL_EPOCAS].where(alcanzan > 0, "no alcanza")
tabla_opt[f"Semillas que alcanzan {UMBRAL_AUC}"] = alcanzan.astype(str) + f"/{len(SEMILLAS)}"
tabla_opt.sort_values("Val AUC", ascending=False)

In [ ]:
#Colores fijos por optimizador (se usan en todas las figuras y en el poster)
COLORES = dict(zip(OPTIMIZADORES, plt.cm.tab10.colors))
VISTOS_EN_CLASE = ["SGD", "Momentum", "Nesterov", "RMSprop", "Adam"]

def curva_media(nombre, metrica):
    #Media y desviacion por epoca de una metrica entre las semillas de un optimizador
    curvas = np.array([c["historia"][metrica] for c in corridas.values() if c["Optimizador"] == nombre])
    return curvas.mean(axis=0), curvas.std(axis=0)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for nombre in OPTIMIZADORES:
    estilo = "-" if nombre in VISTOS_EN_CLASE else "--"
    for a, metrica in zip(ax, ["val_auc", "val_loss"]):
        media, std = curva_media(nombre, metrica)
        epocas = np.arange(1, len(media) + 1)
        a.plot(epocas, media, estilo, color=COLORES[nombre], label=nombre, linewidth=2)
        a.fill_between(epocas, media - std, media + std, color=COLORES[nombre], alpha=0.15)
ax[0].axhline(UMBRAL_AUC, color="gray", linestyle=":", label=f"Umbral AUC = {UMBRAL_AUC}")
ax[0].set_title("ROC-AUC de validación por época")
ax[1].set_title("Pérdida de validación (entropía cruzada) por época")
for a in ax:
    a.set_xlabel("Época")
    a.grid(True)
ax[0].legend(ncol=2)
plt.suptitle("Comparación de optimizadores (media ± desviación de 3 semillas) — línea continua: vistos en clase, discontinua: literatura")
guardar_fig("04_curvas_optimizadores")
plt.show()

In [ ]:
#Barras: AUC de prueba, tiempo y velocidad de convergencia
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
colores = [COLORES[n] for n in OPTIMIZADORES]
graficas = [("Test AUC", "ROC-AUC de prueba"),
            ("Tiempo (s)", "Tiempo de entrenamiento (s)"),
            (COL_EPOCAS, f"Épocas hasta val AUC ≥ {UMBRAL_AUC}")]
for a, (col, titulo) in zip(ax, graficas):
    posiciones = np.arange(len(OPTIMIZADORES))
    a.bar(posiciones, resumen_opt[col]["mean"].fillna(0), yerr=resumen_opt[col]["std"].fillna(0),
          color=colores, capsize=4)
    a.set_xticks(posiciones, OPTIMIZADORES, rotation=45)
    a.set_title(titulo)
    a.grid(True, axis="y")
#Zoom en el eje del AUC para que se noten las diferencias
auc_min = resumen_opt["Test AUC"]["mean"].min()
ax[0].set_ylim(auc_min - 0.02, resumen_opt["Test AUC"]["mean"].max() + 0.01)
#Marcar los optimizadores que no alcanzan el umbral
for i, n in enumerate(OPTIMIZADORES):
    if alcanzan[n] == 0:
        ax[2].text(i, 0, "no\nalcanza", ha="center", va="bottom", color=COLORES[n], fontweight="bold", fontsize=8)
guardar_fig("05_barras_optimizadores")
plt.show()

In [ ]:
#Seleccion del optimizador: mayor AUC de validacion (mejor epoca, promedio de las 3 semillas).
#Se usa VALIDACION, no prueba, para no sesgar la evaluacion final.
ranking = resumen_opt["Val AUC"]["mean"].sort_values(ascending=False)
mejor_opt = ranking.index[0]
top2_opt = ranking.index[:2].tolist()   #se usan como opciones en la busqueda de la Parte 4
print("Ranking (val AUC):")
print(ranking.round(4))
print("\nOptimizador seleccionado:", mejor_opt)
print("Dos mejores (para la Parte 4):", top2_opt)

#Velocidad de convergencia relativa a SGD (evidencia de la necesidad de optimizadores avanzados)
ep_mejor = resumen_opt[COL_EPOCAS]["mean"][mejor_opt]
if alcanzan[mejor_opt] == 0:
    print(f"\nNingún entrenamiento de {mejor_opt} alcanza AUC {UMBRAL_AUC}: ajuste UMBRAL_AUC (no hace falta re-entrenar).")
elif alcanzan["SGD"] == 0:
    print(f"\nSGD no alcanza AUC {UMBRAL_AUC} en {EPOCHS} épocas; {mejor_opt} lo alcanza en {ep_mejor:.1f} épocas.")
else:
    print(f"\n{mejor_opt} alcanza AUC {UMBRAL_AUC} {resumen_opt[COL_EPOCAS]['mean']['SGD'] / ep_mejor:.1f} veces más rápido que SGD.")

### Conclusiones Parte 3

- **Velocidad de convergencia:**
  - Adam, Nadam y AdamW alcanzan un AUC de validación de 0.74 en la **primera época** (en las 3 semillas).
  - RMSprop lo alcanza en ~2.7 épocas.
  - Momentum y Nesterov necesitan **12.3 épocas**, es decir, **≈ 12 veces más**.
  - SGD y Adagrad **no lo alcanzan** en 30 épocas: SGD avanza demasiado lento con lr = 0.001, y Adagrad acumula los gradientes al cuadrado, de modo que su tasa de aprendizaje efectiva decrece hasta estancarse.
- **Desempeño:**
  - El mayor AUC de validación lo obtienen **Adam (0.7490)**, **Nadam (0.7488)** y **AdamW (0.7487)**, que también tienen los mejores AUC de prueba (**0.7488–0.7489**).
  - Sus diferencias (≤ 0.0003) son menores que la desviación entre semillas (≈ 0.0013), por lo que se consideran **equivalentes**.
  - Momentum y Nesterov dan resultados **idénticos** (0.7458 en prueba): con lr = 0.001 la corrección de Nesterov es despreciable.
  - Les siguen RMSprop (0.7446) y Adagrad (0.7357). SGD queda claramente por debajo (0.7171).
- **Costo:**
  - El tiempo de entrenamiento es prácticamente el mismo para todos (≈ 61–65 s por 30 épocas), porque todos hacen el mismo número de actualizaciones.
  - La ventaja real de los adaptativos es que **necesitan muchas menos épocas**.
- **Sobreajuste:**
  - Los adaptativos alcanzan su mejor AUC en la época 5–7 y después el AUC de validación **cae entre 0.009 y 0.016** hasta la época 30: memorizan el conjunto de entrenamiento. Por eso es necesario restaurar la mejor época (o usar *early stopping*) y regularizar.
  - Momentum, Nesterov, Adagrad y SGD tienen su mejor época en la 30 (caída = 0): todavía estaban mejorando, así que con más épocas podrían acercarse a los adaptativos.
- **Selección:** los dos mejores optimizadores por AUC de validación (**Adam y Nadam**) pasan a la búsqueda de hiperparámetros de la Parte 4.
- **Comparación con la referencia:** con hiperparámetros fijos, el mejor MLP (0.7488) **empata** con la regresión logística (0.7497) pero no la supera. La arquitectura fija aprende rápido y se sobreajusta, lo que motiva optimizar la arquitectura, la regularización y la tasa de aprendizaje en la Parte 4.

## PARTE 4: Optimización de hiperparámetros del MLP (Keras Tuner)

En la Parte 3 se fijaron la arquitectura y la tasa de aprendizaje. Ahora se buscan automáticamente los hiperparámetros que **maximizan el AUC de validación**.

**Espacio de búsqueda:**

| Hiperparámetro | Valores |
|---|---|
| Número de capas ocultas | 1 – 4 |
| Neuronas por capa | 32 – 512 (paso 32), una por capa |
| *Dropout* | 0.0 – 0.5 (paso 0.1) |
| Regularización L2 | {0, 1e-5, 1e-4, 1e-3} |
| Tasa de aprendizaje | 1e-4 – 1e-2 (escala logarítmica) |
| Optimizador | los 2 mejores de la Parte 3 |
| *Batch size* | {256, 512, 1024} |

**Estrategias comparadas:**

1. **Random Search:** prueba combinaciones al azar (Bergstra & Bengio, 2012).
2. **Optimización Bayesiana:** un proceso gaussiano propone la siguiente combinación a partir de las anteriores (Snoek et al., 2012).
3. **Hyperband:** entrena muchas configuraciones pocas épocas y solo continúa las prometedoras (Li et al., 2018).

**Presupuesto de búsqueda** (para que quepa en una sesión de Colab):

- La búsqueda usa una **submuestra estratificada de 100.000 registros** del conjunto de entrenamiento.
- Cada prueba tiene como máximo **20 épocas**, con *early stopping* (paciencia 4).
- Random Search y Bayesiana tienen **20 pruebas** cada una.
- Hyperband reparte sus 20 épocas máximas entre varias pruebas.
- Para que la comparación sea justa se mide el **tiempo total** de cada estrategia.

Al final, la mejor configuración de cada estrategia se **re-entrena con todo el conjunto de entrenamiento** y se evalúa en prueba.

In [ ]:
import keras_tuner as kt

#Presupuesto de la busqueda
HPO_MUESTRAS = 100_000
HPO_EPOCHS = 20
MAX_TRIALS = 20

#Submuestra estratificada del conjunto de entrenamiento para la busqueda
x_hpo, _, y_hpo, _ = train_test_split(x_train, y_train, train_size=HPO_MUESTRAS, stratify=y_train, random_state=SEED)
print("Datos para la búsqueda:", x_hpo.shape, f"- incumplimiento: {100 * y_hpo.mean():.2f} %")
print("Optimizadores candidatos (Parte 3):", top2_opt)

#Se guardan los resultados de la parte 3 y se reinicia el diccionario
resultados_opt = dict(resultados)
resultados = {}

In [ ]:
class MLPHiperModelo(kt.HyperModel):
    def build(self, hp):
        #Usamos el modelo Sequential
        model = keras.Sequential()
        model.add(keras.layers.Input(shape=x_train.shape[1:]))
        dropout = hp.Float("dropout", 0.0, 0.5, step=0.1)
        l2 = hp.Choice("l2", [0.0, 1e-5, 1e-4, 1e-3])
        #Numero de capas ocultas: entre 1 y 4, cada una con su numero de neuronas
        for i in range(hp.Int("num_layers", 1, 4)):
            model.add(keras.layers.Dense(units=hp.Int(f"units_{i}", min_value=32, max_value=512, step=32),
                                         activation="relu",
                                         kernel_regularizer=keras.regularizers.l2(l2) if l2 > 0 else None))
            model.add(keras.layers.Dropout(dropout))
        #Capa de salida (clasificacion binaria)
        model.add(keras.layers.Dense(1, activation="sigmoid"))
        #Optimizador y tasa de aprendizaje
        lr = hp.Float("learning_rate", 1e-4, 1e-2, sampling="log")
        optimizador = hp.Choice("optimizer", top2_opt)
        model.compile(optimizer=crear_optimizador(optimizador, lr),
                      loss="binary_crossentropy",
                      metrics=[keras.metrics.AUC(name="auc", num_thresholds=N_UMBRALES),
                               keras.metrics.AUC(curve="PR", name="pr_auc", num_thresholds=N_UMBRALES)])
        return model

    def fit(self, hp, model, *args, **kwargs):
        #El batch size tambien es un hiperparametro
        return model.fit(*args, batch_size=hp.Choice("batch_size", [256, 512, 1024]), **kwargs)

OBJETIVO = kt.Objective("val_auc", direction="max")
parada_temprana = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4)

#Los tuners se guardan en Drive: si Colab se desconecta, con overwrite=False la busqueda continua donde iba.
#(Si se cambia el espacio de busqueda hay que borrar esta carpeta o usar overwrite=True)
TUNER_DIR = os.path.join(RES_DIR, "keras_tuner")
comun = dict(objective=OBJETIVO, seed=SEED, overwrite=False, directory=TUNER_DIR)

tuners = {
    "Random Search": kt.RandomSearch(MLPHiperModelo(), max_trials=MAX_TRIALS, project_name="random_search", **comun),
    "Bayesiana": kt.BayesianOptimization(MLPHiperModelo(), max_trials=MAX_TRIALS, num_initial_points=5,
                                         project_name="bayesiana", **comun),
    "Hyperband": kt.Hyperband(MLPHiperModelo(), max_epochs=HPO_EPOCHS, factor=3, hyperband_iterations=1,
                              project_name="hyperband", **comun),
}
tuners["Random Search"].search_space_summary()

In [ ]:
#Ejecucion de las tres busquedas (el tiempo se acumula en un JSON por si la sesion se reinicia)
ARCHIVO_TIEMPOS = f"{RES_DIR}/parte4_tiempos.json"
tiempos_hpo = {}
if os.path.exists(ARCHIVO_TIEMPOS):
    with open(ARCHIVO_TIEMPOS) as f:
        tiempos_hpo = json.load(f)

for nombre, tuner in tuners.items():
    print(f"\n===== {nombre} =====")
    inicio = time.time()
    tuner.search(x_hpo, y_hpo, epochs=HPO_EPOCHS, validation_data=(x_valid, y_valid),
                 class_weight=class_weight, callbacks=[parada_temprana], verbose=0)
    tiempos_hpo[nombre] = tiempos_hpo.get(nombre, 0) + time.time() - inicio
    with open(ARCHIVO_TIEMPOS, "w") as f:
        json.dump(tiempos_hpo, f)
    print(f"Tiempo de búsqueda: {tiempos_hpo[nombre] / 60:.1f} min")
    tuner.results_summary(num_trials=3)

### Comparación de las estrategias de búsqueda

In [ ]:
def historial_busqueda(tuner):
    #Puntaje (mejor val AUC de la prueba) de cada prueba, en el orden en que se ejecutaron
    filas = []
    for trial_id in sorted(tuner.oracle.trials):
        trial = tuner.oracle.trials[trial_id]
        if trial.score is None:
            continue
        filas.append({"Prueba": len(filas) + 1, "val AUC": trial.score})
    h = pd.DataFrame(filas)
    h["Mejor val AUC acumulado"] = h["val AUC"].cummax()
    return h

historiales = {nombre: historial_busqueda(t) for nombre, t in tuners.items()}

tabla_busqueda = pd.DataFrame({
    nombre: {"Pruebas": len(h),
             "Tiempo (min)": tiempos_hpo[nombre] / 60,
             "Tiempo por prueba (s)": tiempos_hpo[nombre] / len(h),
             "Mejor val AUC (submuestra)": h["val AUC"].max(),
             "Prueba en que se encontró": int(h.loc[h["val AUC"].idxmax(), "Prueba"])}
    for nombre, h in historiales.items()}).T
tabla_busqueda = tabla_busqueda.astype({"Pruebas": int, "Prueba en que se encontró": int})
tabla_busqueda

In [ ]:
#Referencia justa: MLP base (hiperparametros fijos de la Parte 2) evaluado con el MISMO protocolo que las pruebas
#(submuestra, maximo HPO_EPOCHS epocas, early stopping y puntaje = mejor val AUC)
keras.backend.clear_session()
mlp_base_hpo = construir_mlp(keras.optimizers.Adam(learning_rate=LR))
h_base_hpo = mlp_base_hpo.fit(x_hpo, y_hpo, epochs=HPO_EPOCHS, batch_size=BATCH_SIZE,
                              validation_data=(x_valid, y_valid), class_weight=class_weight,
                              callbacks=[keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4)],
                              verbose=0)
auc_base_hpo = max(h_base_hpo.history["val_auc"])
print(f"MLP base (misma submuestra): val AUC = {auc_base_hpo:.4f}")

#Convergencia de cada estrategia: mejor val AUC encontrado vs numero de prueba, y mejor AUC final con su costo
COLORES_HPO = {"Random Search": "tab:blue", "Bayesiana": "tab:red", "Hyperband": "tab:green"}
fig, ax = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [2, 1]})
for nombre, h in historiales.items():
    ax[0].step(h["Prueba"], h["Mejor val AUC acumulado"], where="post", color=COLORES_HPO[nombre], label=nombre, linewidth=2)
    ax[0].scatter(h["Prueba"], h["val AUC"], color=COLORES_HPO[nombre], alpha=0.35, s=20)
ax[0].set_title("Mejor val AUC encontrado vs. número de prueba\n(puntos: AUC de cada prueba)")
ax[0].set_xlabel("Prueba")

nombres = list(historiales)
mejores_auc = tabla_busqueda["Mejor val AUC (submuestra)"].astype(float)
ax[1].bar(nombres, mejores_auc, color=[COLORES_HPO[n] for n in nombres])
for i, n in enumerate(nombres):
    ax[1].text(i, mejores_auc[n] + 0.0002, f"{mejores_auc[n]:.4f}\n{int(tabla_busqueda.loc[n, 'Pruebas'])} pruebas\n"
               f"{float(tabla_busqueda.loc[n, 'Tiempo (min)']):.1f} min", ha="center", va="bottom", fontsize=9)
ax[1].set_title("Mejor val AUC y costo de cada estrategia")

#Zoom en la zona de las mejores pruebas (las peores pueden tener AUC mucho mas bajo)
acumulado = pd.concat(historiales.values())["Mejor val AUC acumulado"]
abajo = min(acumulado.min(), auc_base_hpo) - 0.005
arriba = max(acumulado.max(), auc_base_hpo) + 0.004
for a in ax:
    a.axhline(auc_base_hpo, color="gray", linestyle=":", label="MLP base (misma submuestra)")
    a.set_ylabel("ROC-AUC de validación")
    a.set_ylim(abajo, arriba if a is ax[0] else arriba + 0.003)
    a.grid(True, axis="y")
ax[0].grid(True)
ax[0].legend()
guardar_fig("06_hpo_convergencia")
plt.show()

In [ ]:
#Mejores hiperparametros de cada estrategia
def hp_legibles(hp):
    v = dict(hp.values)
    n = v["num_layers"]
    v["arquitectura"] = "-".join(str(v[f"units_{i}"]) for i in range(n))
    #Las neuronas de capas que no existen (i >= num_layers) son residuos de otras pruebas y no se usan
    return {k: v[k] for k in ["num_layers", "arquitectura", "dropout", "l2", "learning_rate", "optimizer", "batch_size"]}

mejores_hps = {nombre: t.get_best_hyperparameters(num_trials=1)[0] for nombre, t in tuners.items()}
pd.DataFrame({nombre: hp_legibles(hp) for nombre, hp in mejores_hps.items()}).T

### ¿Por qué no un Grid Search?

Si se discretiza el mismo espacio y se prueban todas las combinaciones (*Grid Search*), con 5 valores de tasa de aprendizaje:

In [ ]:
n_units = len(range(32, 513, 32))                             #16 opciones de neuronas por capa
arquitecturas = sum(n_units ** k for k in range(1, 5))        #1 a 4 capas
combinaciones = arquitecturas * 6 * 4 * 5 * len(top2_opt) * 3 #dropout, l2, lr, optimizador, batch size
seg_por_prueba = float(tabla_busqueda.loc["Random Search", "Tiempo por prueba (s)"])
horas_grid = combinaciones * seg_por_prueba / 3600

print(f"Combinaciones de un Grid Search: {combinaciones:,}")
print(f"Tiempo promedio por prueba (submuestra, con early stopping): {seg_por_prueba:.1f} s")
print(f"Tiempo estimado del Grid Search: {horas_grid:,.0f} horas (≈ {horas_grid / 24 / 365:,.0f} años)")
print(f"Pruebas usadas por Random Search / Bayesiana: {MAX_TRIALS} "
      f"(≈ {100 * MAX_TRIALS / combinaciones:.6f} % del espacio), en ≈ {tiempos_hpo['Random Search'] / 60:.0f} min")

### Re-entrenamiento de las mejores configuraciones con todos los datos de entrenamiento

In [ ]:
def entrenar_mejor(nombre, tuner, hp):
    #Reconstruye el modelo con los mejores hiperparametros y lo entrena con todo el conjunto de entrenamiento
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = tuner.hypermodel.build(hp)
    parada = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5)
    inicio = time.time()
    history = model.fit(x_train, y_train, epochs=EPOCHS, batch_size=hp.get("batch_size"),
                        validation_data=(x_valid, y_valid), class_weight=class_weight,
                        callbacks=[parada, MejoresPesos()], verbose=0)
    evaluar(f"MLP optimizado ({nombre})", model, history, time.time() - inicio)
    return model, history

modelos_hpo, historias_hpo = {}, {}
for nombre, tuner in tuners.items():
    modelos_hpo[nombre], historias_hpo[nombre] = entrenar_mejor(nombre, tuner, mejores_hps[nombre])

In [ ]:
#Comparacion: MLP base vs. mejor optimizador de la Parte 3 vs. MLP con hiperparametros optimizados
COLS_HP = ["Tiempo (s)", "Val AUC", "Test AUC", "Test PR-AUC", "Test Recall", "Test F1"]
fila_opt = tabla_corridas[tabla_corridas["Optimizador"] == mejor_opt][COLS_HP].mean()
tabla_hp = pd.concat([
    pd.DataFrame(resultados_base).T.loc[["MLP base (Adam)"], COLS_HP],
    pd.DataFrame({f"MLP con {mejor_opt} (Parte 3, media 3 semillas)": fila_opt}).T,
    pd.DataFrame(resultados).T[COLS_HP],
]).astype(float)
tabla_hp["Mejora AUC vs base"] = tabla_hp["Test AUC"] - tabla_hp.loc["MLP base (Adam)", "Test AUC"]
tabla_hp.round(4)

In [ ]:
#Modelo final de la Parte 4: el de mayor AUC de VALIDACION
mejor_hpo = pd.DataFrame(resultados).T["Val AUC"].astype(float).idxmax()
estrategia = mejor_hpo.replace("MLP optimizado (", "").rstrip(")")
mejor_mlp = modelos_hpo[estrategia]
print("Modelo seleccionado:", mejor_hpo)
print(hp_legibles(mejores_hps[estrategia]))
mejor_mlp.summary()
graficar(historias_hpo[estrategia], f"Curva de aprendizaje - {mejor_hpo}")

### Conclusiones Parte 4

- **Comparación de estrategias** (submuestra de 100.000 registros, costo similar de ≈ 9.2 min cada una):
  - La **Bayesiana** encontró el mejor AUC (**0.7476**, prueba 18), seguida de cerca por **Hyperband** (0.7475, prueba 29) y **Random Search** (0.7464, prueba 9).
  - **Hyperband evaluó 30 configuraciones** en el mismo tiempo en que las otras evaluaron 20, porque descarta pronto las que no prometen.
  - Las diferencias entre estrategias son pequeñas (≈ 0.001): con 20–30 pruebas en un espacio de 7 hiperparámetros, las tres llegan a la misma zona.
- **Configuración ganadora:** las tres estrategias coinciden en lo esencial:
  - **2 capas ocultas**, *dropout* alto (0.3–0.4) y la **regularización L2 más fuerte** del espacio (1e-3).
  - Una tasa de aprendizaje **menor** que la del modelo base (2.5e-4 a 4.8e-4 frente a 1e-3). Nadam fue el optimizador elegido en 2 de 3 estrategias.
  - Esto confirma el diagnóstico de las Partes 2 y 3: el problema necesita **más regularización y pasos más pequeños** para no sobreajustar.
- **Mejora frente al modelo sin optimizar:** al re-entrenar con todos los datos, el MLP optimizado obtiene un AUC de prueba de **0.7496** (Random Search) frente a 0.7494 del MLP base: la mejora es **prácticamente nula (+0.0002)**.
  - El MLP se satura en ≈ 0.75, el mismo nivel de la regresión logística: los hiperparámetros **no son el cuello de botella**, sino el tipo de modelo para datos tabulares. Esto motiva la Parte 5.
  - Los modelos optimizados sí son **más eficientes**: llegan al mismo AUC con redes regularizadas que, gracias al *early stopping*, entrenan en 23–32 s (Bayesiana, Hyperband) frente a 69 s del MLP base.
- **Necesidad de optimización:** un Grid Search sobre el mismo espacio tendría **≈ 50 millones de combinaciones** y, a ≈ 28 s por prueba, tardaría **≈ 45 años**. Con 20 pruebas guiadas (≈ 9 min) se exploró el espacio lo suficiente para concluir que el MLP no puede superar ≈ 0.75 en este problema.

## PARTE 5: LightGBM + Optuna

Los modelos de *gradient boosting* sobre árboles de decisión, como **LightGBM** (Ke et al., 2017), son el estado del arte en datos tabulares y fueron la base de las mejores soluciones de esta competencia en Kaggle. A diferencia del MLP, no se entrenan con descenso por gradiente sobre pesos: en cada iteración se agrega un árbol que corrige los errores de los anteriores. Pero su desempeño depende **mucho** de sus hiperparámetros.

Se compara:

1. **LightGBM con hiperparámetros por defecto** (100 árboles, tasa de aprendizaje 0.1, 31 hojas).
2. **LightGBM optimizado con Optuna** (Akiba et al., 2019), con dos estrategias y el mismo presupuesto de **30 pruebas**:
   - **TPE** (*Tree-structured Parzen Estimator*): optimización bayesiana que modela qué regiones del espacio dan buenos resultados (Bergstra et al., 2011). Sus primeras 5 pruebas son aleatorias y, como se usa la misma semilla, coinciden con las de la búsqueda aleatoria; a partir de la sexta, TPE propone las combinaciones con su modelo.
   - **Búsqueda aleatoria** (*RandomSampler*).

**Espacio de búsqueda:**

| Hiperparámetro | Valores | Qué controla |
|---|---|---|
| `learning_rate` | 0.02 – 0.2 (log) | tamaño del paso de cada árbol |
| `num_leaves` | 16 – 256 (log) | complejidad de cada árbol |
| `max_depth` | 3 – 12 | profundidad máxima |
| `min_child_samples` | 10 – 300 (log) | mínimo de registros por hoja (regularización) |
| `subsample` | 0.5 – 1.0 | fracción de registros por árbol |
| `colsample_bytree` | 0.2 – 1.0 | fracción de variables por árbol |
| `reg_alpha`, `reg_lambda` | 1e-8 – 10 (log) | regularización L1 y L2 |

**Condiciones de la búsqueda** (iguales a las de la Parte 4):

- El número de árboles se fija con *early stopping* sobre validación (máximo 2000, paciencia 100).
- Se usa la misma submuestra de 100.000 registros.
- La mejor configuración de cada estrategia se re-entrena con todo el conjunto de entrenamiento.

In [ ]:
import lightgbm as lgb
import optuna
import shutil
import tempfile
optuna.logging.set_verbosity(optuna.logging.WARNING)

#Se guardan los resultados de la parte 4 y se reinicia el diccionario
resultados_hpo = dict(resultados)
resultados = {}

N_TRIALS_LGB = 30
MAX_ARBOLES = 2000

def entrenar_lgb(params, x, y):
    #LightGBM con early stopping sobre validacion (el numero de arboles lo decide la parada temprana)
    model = lgb.LGBMClassifier(n_estimators=MAX_ARBOLES, class_weight="balanced", subsample_freq=1,
                               random_state=SEED, n_jobs=-1, verbose=-1, **params)
    model.fit(x, y, eval_set=[(x_valid, y_valid)], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, verbose=False)])
    return model

### LightGBM con hiperparámetros por defecto

In [ ]:
inicio = time.time()
lgb_default = lgb.LGBMClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)
lgb_default.fit(x_train, y_train)
evaluar("LightGBM (por defecto)", lgb_default, tiempo=time.time() - inicio)

### Búsqueda de hiperparámetros con Optuna

Cada estudio se guarda en una base de datos SQLite que se respalda en Drive después de cada prueba. Si Colab se desconecta, al volver a ejecutar la celda **solo se corren las pruebas que faltan**.

In [ ]:
def objetivo(trial):
    #Funcion objetivo: AUC de validacion de LightGBM entrenado con la submuestra
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 300, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    model = entrenar_lgb(params, x_hpo, y_hpo)
    trial.set_user_attr("arboles", int(model.best_iteration_))
    return metrics.roc_auc_score(y_valid, model.predict_proba(x_valid)[:, 1])

#La base de datos se usa en una carpeta local (SQLite no funciona bien directamente sobre Drive)
#y se copia a Drive despues de cada prueba
OPTUNA_DIR = os.path.join(RES_DIR, "optuna")
os.makedirs(OPTUNA_DIR, exist_ok=True)

def crear_estudio(archivo, sampler):
    local = os.path.join(tempfile.gettempdir(), f"optuna_{archivo}.db")
    respaldo = os.path.join(OPTUNA_DIR, f"{archivo}.db")
    if os.path.exists(respaldo) and not os.path.exists(local):
        shutil.copy(respaldo, local)
    estudio = optuna.create_study(study_name=archivo, storage=f"sqlite:///{local}", load_if_exists=True,
                                  direction="maximize", sampler=sampler)
    return estudio, local, respaldo

#TPE explora al azar sus primeras 5 pruebas (como num_initial_points=5 en la Parte 4) y luego usa el modelo
ESTUDIOS = {"TPE": ("tpe", optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5)),
            "Búsqueda aleatoria": ("aleatoria", optuna.samplers.RandomSampler(seed=SEED))}

In [ ]:
estudios = {}
for nombre, (archivo, sampler) in ESTUDIOS.items():
    estudio, local, respaldo = crear_estudio(archivo, sampler)
    completas = len(estudio.get_trials(states=[optuna.trial.TrialState.COMPLETE]))
    faltan = N_TRIALS_LGB - completas
    print(f"\n===== {nombre}: {completas} pruebas guardadas, faltan {max(faltan, 0)} =====")
    if faltan > 0:
        def respaldar(study, trial):
            shutil.copy(local, respaldo)
            print(f"  Prueba {trial.number + 1:2d}: val AUC = {trial.value:.4f} | mejor = {study.best_value:.4f} "
                  f"| árboles = {trial.user_attrs['arboles']}")
        estudio.optimize(objetivo, n_trials=faltan, callbacks=[respaldar])
    estudios[nombre] = estudio

### Comparación de las estrategias de búsqueda

In [ ]:
def historial_optuna(estudio):
    #AUC y duracion de cada prueba completada, en orden de ejecucion
    pruebas = sorted(estudio.get_trials(states=[optuna.trial.TrialState.COMPLETE]), key=lambda t: t.number)
    h = pd.DataFrame({"Prueba": np.arange(1, len(pruebas) + 1),
                      "val AUC": [t.value for t in pruebas],
                      "Duración (s)": [(t.datetime_complete - t.datetime_start).total_seconds() for t in pruebas]})
    h["Mejor val AUC acumulado"] = h["val AUC"].cummax()
    h["Tiempo acumulado (min)"] = h["Duración (s)"].cumsum() / 60
    return h

historiales_lgb = {nombre: historial_optuna(e) for nombre, e in estudios.items()}

tabla_busqueda_lgb = pd.DataFrame({
    nombre: {"Pruebas": len(h),
             "Tiempo (min)": h["Duración (s)"].sum() / 60,
             "Mejor val AUC (submuestra)": h["val AUC"].max(),
             "Prueba en que se encontró": int(h.loc[h["val AUC"].idxmax(), "Prueba"]),
             #Desde la prueba 6 TPE ya usa su modelo: un promedio alto indica que concentra las pruebas en la zona buena
             f"Val AUC promedio pruebas 6–{N_TRIALS_LGB}": h["val AUC"].iloc[5:].mean(),
             f"Desviación pruebas 6–{N_TRIALS_LGB}": h["val AUC"].iloc[5:].std()}
    for nombre, h in historiales_lgb.items()}).T
tabla_busqueda_lgb = tabla_busqueda_lgb.astype({"Pruebas": int, "Prueba en que se encontró": int})
tabla_busqueda_lgb

In [ ]:
#Convergencia: mejor val AUC encontrado vs numero de prueba y vs tiempo de busqueda
COLORES_LGB = {"TPE": "tab:red", "Búsqueda aleatoria": "tab:blue"}
#La busqueda aleatoria se dibuja discontinua y TPE encima (sus primeras 5 pruebas coinciden)
ESTILO_LGB = {"TPE": dict(linestyle="-", linewidth=2.5, zorder=3), "Búsqueda aleatoria": dict(linestyle="--", linewidth=2, zorder=2)}
#Referencia justa: LightGBM por defecto entrenado con la MISMA submuestra que usan las busquedas
lgb_default_hpo = lgb.LGBMClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1).fit(x_hpo, y_hpo)
auc_default = metrics.roc_auc_score(y_valid, lgb_default_hpo.predict_proba(x_valid)[:, 1])
print(f"LightGBM por defecto (submuestra): val AUC = {auc_default:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for nombre, h in historiales_lgb.items():
    ax[0].step(h["Prueba"], h["Mejor val AUC acumulado"], where="post", color=COLORES_LGB[nombre], label=nombre, **ESTILO_LGB[nombre])
    ax[0].scatter(h["Prueba"], h["val AUC"], color=COLORES_LGB[nombre], alpha=0.35, s=20)
    ax[1].step(h["Tiempo acumulado (min)"], h["Mejor val AUC acumulado"], where="post", color=COLORES_LGB[nombre],
               label=nombre, **ESTILO_LGB[nombre])
ax[0].set_title("Mejor val AUC encontrado vs. número de prueba\n(puntos: AUC de cada prueba)")
ax[0].set_xlabel("Prueba")
ax[1].set_title("Mejor val AUC encontrado vs. tiempo de búsqueda")
ax[1].set_xlabel("Tiempo acumulado (min)")
todos = pd.concat(historiales_lgb.values())
for a in ax:
    a.axhline(auc_default, color="gray", linestyle=":", label="LightGBM por defecto (misma submuestra)")
    a.set_ylabel("ROC-AUC de validación")
    a.set_ylim(min(todos["Mejor val AUC acumulado"].min(), auc_default) - 0.005, max(todos["val AUC"].max(), auc_default) + 0.003)
    a.grid(True)
    a.legend()
guardar_fig("07_optuna_convergencia")
plt.show()

In [ ]:
#Mejores hiperparametros de cada estrategia
pd.DataFrame({nombre: {**e.best_params, "árboles (early stopping)": e.best_trial.user_attrs["arboles"]}
              for nombre, e in estudios.items()}).T

In [ ]:
#Importancia de cada hiperparametro en el AUC (fANOVA de Optuna)
importancia_hp = pd.DataFrame({nombre: optuna.importance.get_param_importances(
                                   e, evaluator=optuna.importance.FanovaImportanceEvaluator(seed=SEED))
                               for nombre, e in estudios.items()}).fillna(0)
importancia_hp = importancia_hp.sort_values("TPE")
importancia_hp.plot(kind="barh", figsize=(8, 5), color=[COLORES_LGB[c] for c in importancia_hp.columns])
plt.title("Importancia de los hiperparámetros de LightGBM")
plt.xlabel("Importancia relativa")
plt.grid(True, axis="x")
guardar_fig("08_optuna_importancia_hp")
plt.show()

### Re-entrenamiento de las mejores configuraciones con todos los datos de entrenamiento

In [ ]:
modelos_lgb = {"LightGBM (por defecto)": lgb_default}
for nombre, estudio in estudios.items():
    inicio = time.time()
    model = entrenar_lgb(estudio.best_params, x_train, y_train)
    clave = f"LightGBM optimizado ({nombre})"
    evaluar(clave, model, tiempo=time.time() - inicio)
    print(f"  árboles: {model.best_iteration_}")
    modelos_lgb[clave] = model

In [ ]:
#Comparacion: regresion logistica, mejor MLP (Parte 4) y LightGBM por defecto vs optimizado
COLS_LGB = ["Tiempo (s)", "Val AUC", "Test AUC", "Test PR-AUC", "Test Recall", "Test F1"]
tabla_lgb = pd.concat([
    pd.DataFrame(resultados_base).T.loc[["Regresión logística"], COLS_LGB],
    pd.DataFrame(resultados_hpo).T.loc[[mejor_hpo], COLS_LGB],
    pd.DataFrame(resultados).T[COLS_LGB],
]).astype(float)
tabla_lgb["Mejora AUC vs LightGBM por defecto"] = tabla_lgb["Test AUC"] - tabla_lgb.loc["LightGBM (por defecto)", "Test AUC"]
tabla_lgb.round(4)

In [ ]:
#Modelo final de la Parte 5: el de mayor AUC de VALIDACION
mejor_lgb_nombre = pd.DataFrame(resultados).T["Val AUC"].astype(float).idxmax()
mejor_lgb = modelos_lgb[mejor_lgb_nombre]
print("Modelo seleccionado:", mejor_lgb_nombre)

#Variables mas importantes del modelo (ganancia total en las divisiones de los arboles)
importancia_var = pd.Series(mejor_lgb.booster_.feature_importance(importance_type="gain"), index=feature_names)
importancia_var = (importancia_var / importancia_var.sum()).sort_values().tail(15)
plt.figure(figsize=(8, 6))
importancia_var.plot(kind="barh", color="tab:green")
plt.title(f"Top 15 variables más importantes\n{mejor_lgb_nombre}")
plt.xlabel("Proporción de la ganancia total")
plt.grid(True, axis="x")
guardar_fig("09_importancia_variables")
plt.show()

### Conclusiones Parte 5

- **LightGBM supera a todos los modelos anteriores**, incluso sin optimizar: con hiperparámetros por defecto obtiene un AUC de prueba de **0.7652**, frente a 0.7497 de la regresión logística y 0.7496 del mejor MLP (+0.0155).
- **Efecto de la optimización:**
  - En la submuestra, la búsqueda sube el AUC de validación de **0.7561** (por defecto) a **0.7637** (+0.0076).
  - Re-entrenado con todos los datos, el LightGBM optimizado alcanza un AUC de prueba de **0.7696** (TPE) y 0.7697 (búsqueda aleatoria), **+0.0044** frente al de por defecto.
  - También mejoran la PR-AUC (0.2535 → 0.2594) y el F1 (0.3109 → 0.3196). A diferencia del MLP, en LightGBM **la optimización de hiperparámetros sí mejora el modelo**.
- **TPE vs búsqueda aleatoria:**
  - El mejor valor es casi igual (0.7637 TPE en la prueba 23 vs 0.7634 aleatoria en la prueba 27). Ambas encuentran una configuración buena ya en la prueba 2, que comparten porque las 5 primeras pruebas de TPE son aleatorias.
  - La diferencia está en **dónde** prueba cada una: a partir de la prueba 6, TPE concentra sus pruebas en la zona buena (casi todas entre 0.759 y 0.764), mientras que la búsqueda aleatoria sigue dispersa (0.750–0.763). Ver la columna de promedio y desviación de las pruebas 6–30.
  - **Costo:** TPE tardó 20.1 min frente a 12.4 min de la aleatoria, porque se concentra en configuraciones más complejas (más árboles y hojas), que son más lentas de entrenar.
- **Hiperparámetros:**
  - Ambas estrategias eligen tasas de aprendizaje bajas (0.02–0.04) con muchos árboles (≈ 850–935 en la submuestra, ≈ 1200 con todos los datos), árboles poco profundos (`max_depth` 4–8), muchos registros por hoja (`min_child_samples` 104–148) y solo ≈ 40 % de las variables por árbol: **modelos muy regularizados**.
  - Según la búsqueda aleatoria, que muestrea todo el espacio de manera uniforme, el hiperparámetro más influyente es la **tasa de aprendizaje** (importancia 0.81). En TPE pesan más `min_child_samples` y `max_depth`, porque sus pruebas ya se concentran en la región de tasas de aprendizaje buenas.
- **Interpretación:**
  - Las variables más importantes son las puntuaciones externas (`EXT_SOURCE_MEAN`, `EXT_SOURCE_3`, `EXT_SOURCE_2`, `EXT_SOURCE_1`), el plazo del crédito (`CREDIT_TERM`) y la edad (`DAYS_BIRTH`).
  - **Dos de las tres más importantes (`EXT_SOURCE_MEAN` y `CREDIT_TERM`) son variables creadas en la Parte 1**, lo que valida la ingeniería de variables.

## PARTE 6: Evaluación final y comparación global

Se comparan en el **conjunto de prueba** (46.127 clientes que ningún modelo vio durante el entrenamiento ni la selección) los modelos finales de cada etapa:

| Modelo | Optimización aplicada |
|---|---|
| Regresión logística | Ninguna (referencia) |
| MLP base | Ninguna (hiperparámetros fijos, Adam) |
| MLP optimizado | Keras Tuner (mejor estrategia de la Parte 4) |
| LightGBM por defecto | Ninguna |
| LightGBM optimizado | Optuna (mejor estrategia de la Parte 5) |

Además del AUC, se mide qué tan útil sería cada modelo para el banco: **qué porcentaje de los clientes que incumplen queda dentro del 10 % y del 20 % de clientes con mayor riesgo predicho**.

In [ ]:
#Modelos finales de cada etapa y su optimizacion
modelos_finales = {
    "Regresión logística": (log_reg, "Ninguna (referencia)"),
    "MLP base (Adam)": (model_base, "Ninguna (hiperparámetros fijos)"),
    mejor_hpo: (mejor_mlp, f"Keras Tuner – {estrategia}"),
    "LightGBM (por defecto)": (lgb_default, "Ninguna"),
    mejor_lgb_nombre: (mejor_lgb, "Optuna – " + mejor_lgb_nombre.replace("LightGBM optimizado (", "").rstrip(")")),
}
#Resultados calculados en las partes anteriores (mismo conjunto de prueba y mismo criterio de umbral)
todos_resultados = {**resultados_base, **resultados_hpo, **resultados}

#Probabilidades en prueba de cada modelo
p_test = {nombre: predecir_proba(m, x_test) for nombre, (m, _) in modelos_finales.items()}

def captura(p, fraccion):
    #Porcentaje de los incumplidos que queda dentro de la fraccion de clientes con mayor riesgo predicho
    n = int(len(p) * fraccion)
    mas_riesgosos = np.argsort(-p)[:n]
    return y_test[mas_riesgosos].sum() / y_test.sum()

COLS_FINAL = ["Val AUC", "Test AUC", "Test PR-AUC", "Umbral", "Test Precision", "Test Recall", "Test F1", "Tiempo (s)"]
tabla_final = pd.DataFrame({n: todos_resultados[n] for n in modelos_finales}).T[COLS_FINAL].astype(float)
tabla_final.insert(0, "Optimización", [opt for _, opt in modelos_finales.values()])
tabla_final["Incumplidos en el 10 % más riesgoso"] = [captura(p_test[n], 0.10) for n in modelos_finales]
tabla_final["Incumplidos en el 20 % más riesgoso"] = [captura(p_test[n], 0.20) for n in modelos_finales]
tabla_final["Mejora AUC vs regresión logística"] = tabla_final["Test AUC"] - tabla_final.loc["Regresión logística", "Test AUC"]
tabla_final.to_csv(f"{RES_DIR}/tabla_final.csv")
tabla_final.round(4)

In [ ]:
#Curvas ROC y Precision-Recall de los modelos finales en el conjunto de prueba
COLORES_FINAL = dict(zip(modelos_finales, ["tab:gray", "tab:orange", "tab:red", "tab:cyan", "tab:green"]))
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
for nombre, p in p_test.items():
    fpr, tpr, _ = metrics.roc_curve(y_test, p)
    ax[0].plot(fpr, tpr, color=COLORES_FINAL[nombre], linewidth=2,
               label=f"{nombre} (AUC = {metrics.roc_auc_score(y_test, p):.4f})")
    prec, rec, _ = metrics.precision_recall_curve(y_test, p)
    ax[1].plot(rec, prec, color=COLORES_FINAL[nombre], linewidth=2,
               label=f"{nombre} (PR-AUC = {metrics.average_precision_score(y_test, p):.4f})")
ax[0].plot([0, 1], [0, 1], "k--", label="Azar (AUC = 0.5)")
ax[0].set_title("Curvas ROC – conjunto de prueba")
ax[0].set_xlabel("Tasa de falsos positivos")
ax[0].set_ylabel("Tasa de verdaderos positivos (recall)")
ax[1].axhline(y_test.mean(), color="k", linestyle="--", label=f"Azar (precisión = {y_test.mean():.3f})")
ax[1].set_title("Curvas Precision-Recall – conjunto de prueba")
ax[1].set_xlabel("Recall (incumplidos detectados)")
ax[1].set_ylabel("Precisión")
for a in ax:
    a.grid(True)
    a.legend(loc="lower right" if a is ax[0] else "upper right", fontsize=9)
guardar_fig("10_curvas_roc")
plt.show()

In [ ]:
#Resumen visual: AUC de prueba y captura de incumplidos de cada modelo final
import matplotlib
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
nombres = list(modelos_finales)
colores = [COLORES_FINAL[n] for n in nombres]
etiquetas = [n.replace(" optimizado", "\noptimizado").replace(" (", "\n(") for n in nombres]
graficas = [("Test AUC", "ROC-AUC de prueba", "{:.4f}"),
            ("Incumplidos en el 20 % más riesgoso", "Incumplidos detectados en el 20 % de clientes más riesgosos", "{:.1%}")]
for a, (col, titulo, fmt) in zip(ax, graficas):
    valores = tabla_final[col]
    a.bar(range(len(nombres)), valores, color=colores)
    for i, v in enumerate(valores):
        a.text(i, v, fmt.format(v), ha="center", va="bottom", fontsize=10)
    a.set_xticks(range(len(nombres)), etiquetas, fontsize=9)
    a.set_title(titulo)
    a.set_ylim(valores.min() - 0.02, valores.max() + 0.01)
    a.grid(True, axis="y")
ax[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
guardar_fig("12_resumen_modelos")
plt.show()

In [ ]:
#Matriz de confusion del mejor modelo (mayor AUC de VALIDACION) con el umbral elegido en validacion
mejor_final = tabla_final["Val AUC"].idxmax()
umbral = tabla_final.loc[mejor_final, "Umbral"]
y_pred = (p_test[mejor_final] >= umbral).astype(int)
cm = metrics.confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
etiquetas_cm = ["Paga (0)", "Incumple (1)"]
metrics.ConfusionMatrixDisplay(cm, display_labels=etiquetas_cm).plot(ax=ax[0], cmap="Blues", values_format=",d", colorbar=False)
ax[0].set_title(f"Matriz de confusión – {mejor_final}\n(umbral = {umbral:.3f}, elegido en validación)")
metrics.ConfusionMatrixDisplay(cm / cm.sum(axis=1, keepdims=True), display_labels=etiquetas_cm).plot(
    ax=ax[1], cmap="Blues", values_format=".1%", colorbar=False)
ax[1].set_title("Proporción por clase real")
for a in ax:
    a.set_xlabel("Predicción")
    a.set_ylabel("Clase real")
guardar_fig("11_matriz_confusion")
plt.show()

print(f"Modelo: {mejor_final}")
print(f"Incumplidos en prueba: {tp + fn:,} de {len(y_test):,}")
print(f"  Detectados (verdaderos positivos): {tp:,} ({tp / (tp + fn):.1%} de los incumplidos)")
print(f"  No detectados (falsos negativos):  {fn:,}")
print(f"Clientes señalados como riesgosos: {tp + fp:,} ({(tp + fp) / len(y_test):.1%} del total)")
print(f"  De cada 100 señalados, {100 * tp / (tp + fp):.0f} realmente incumplen "
      f"(la tasa base es {100 * y_test.mean():.0f} de cada 100)")
print(f"Buenos clientes señalados por error (falsos positivos): {fp:,} ({fp / (tn + fp):.1%} de los que pagan)")

In [ ]:
#Resumen del efecto de cada tecnica de optimizacion sobre el AUC de prueba
efecto_opt = pd.DataFrame([
    {"Etapa": "Parte 3 – Optimizador de entrenamiento (SGD → " + mejor_opt + ")",
     "AUC prueba antes": resumen_opt["Test AUC"]["mean"]["SGD"],
     "AUC prueba después": resumen_opt["Test AUC"]["mean"][mejor_opt]},
    {"Etapa": "Parte 4 – Hiperparámetros del MLP (" + estrategia + ")",
     "AUC prueba antes": resultados_base["MLP base (Adam)"]["Test AUC"],
     "AUC prueba después": resultados_hpo[mejor_hpo]["Test AUC"]},
    {"Etapa": "Parte 5 – Hiperparámetros de LightGBM (" + mejor_lgb_nombre.replace("LightGBM optimizado (", "").rstrip(")") + ")",
     "AUC prueba antes": resultados["LightGBM (por defecto)"]["Test AUC"],
     "AUC prueba después": resultados[mejor_lgb_nombre]["Test AUC"]},
    {"Etapa": "Global – regresión logística → mejor modelo",
     "AUC prueba antes": tabla_final.loc["Regresión logística", "Test AUC"],
     "AUC prueba después": tabla_final.loc[mejor_final, "Test AUC"]},
]).set_index("Etapa")
efecto_opt["Mejora"] = efecto_opt["AUC prueba después"] - efecto_opt["AUC prueba antes"]
efecto_opt.round(4)

### Conclusiones Parte 6

- **Mejor modelo:** **LightGBM optimizado con TPE**, con AUC de prueba **0.7696**, +0.020 frente a la regresión logística (≈ 0.750). Su PR-AUC (**0.2594**) es ≈ 11 % mayor que la de la regresión logística (0.2331), es decir, separa mejor a los clientes que incumplen dentro de una clase muy minoritaria.
- **Utilidad práctica** (umbral 0.668, elegido en validación para maximizar F1):
  - El modelo detecta **1.561 de los 3.724 incumplidos** del conjunto de prueba (**41.9 %**) señalando solo al **13.1 %** de los clientes.
  - **De cada 100 clientes señalados, 26 realmente incumplen**, frente a 8 de cada 100 si se eligieran al azar: el modelo concentra el riesgo ≈ 3.2 veces.
  - El costo es que el 10.6 % de los buenos clientes (4.483) quedan señalados por error. El umbral se puede mover según el costo que el banco le asigne a cada tipo de error.
  - Si se revisa el **20 % de clientes con mayor riesgo predicho**, se encuentra el **53.9 % de todos los incumplidos** (50.7 % con la regresión logística); en el 10 % más riesgoso, el 35.5 % (32.8 % con la regresión logística).
- **Validación vs prueba:** en todos los modelos el AUC de validación y el de prueba difieren en menos de 0.002. Elegir modelos y umbrales con el conjunto de validación **no sobreestimó** el desempeño, y el AUC de prueba es una estimación confiable de cómo funcionaría el modelo con clientes nuevos.


## Conclusiones generales

1. **La optimización del entrenamiento es indispensable:**
   - Con la misma red y el mismo costo por época, Adam, Nadam y AdamW llegan a un AUC de validación de 0.74 en **1 época**, Momentum/Nesterov en 12, y SGD y Adagrad no lo logran en 30.
   - Elegir bien el optimizador sube el AUC de prueba de **0.717 (SGD) a 0.749 (Adam)**.
2. **Los optimizadores adaptativos convergen rápido pero sobreajustan.** Su mejor época está entre la 5 y la 7, y después el AUC de validación cae. Restaurar la mejor época (*early stopping*) y regularizar es parte del proceso de optimización.
3. **La optimización de hiperparámetros depende del modelo:**
   - **MLP:** Random Search, Bayesiana e Hyperband coinciden en una red de 2 capas muy regularizada (L2 = 1e-3, *dropout* 0.3–0.4, lr ≈ 3e-4), pero el AUC de prueba casi no cambia (0.7494 → 0.7496). El MLP se satura al nivel de la regresión logística (≈ 0.75).
   - **LightGBM:** la búsqueda con Optuna sí mejora el modelo (0.7652 → **0.7696**, y PR-AUC 0.2535 → 0.2594).
4. **Las búsquedas inteligentes hacen viable la optimización:**
   - Un Grid Search sobre el espacio del MLP tendría ≈ 50 millones de combinaciones y tardaría ≈ 45 años; con 20–30 pruebas (≈ 9–20 min) se llegó a la zona óptima.
   - Las estrategias bayesianas (Bayesiana con proceso gaussiano y TPE) encontraron los mejores valores y concentran sus pruebas en la zona buena. Hyperband evaluó un 50 % más de configuraciones con el mismo tiempo.
   - Con espacios moderados y suficientes pruebas, la búsqueda aleatoria queda muy cerca.
5. **Mejor modelo:** **LightGBM optimizado con TPE**, con AUC de prueba **0.7696**, +0.020 frente a la regresión logística. En datos tabulares el *gradient boosting* supera a las redes neuronales, incluso cuando estas se optimizan.
6. **Interpretabilidad:** las puntuaciones externas de crédito (`EXT_SOURCE_*`), el plazo del crédito y la edad son las variables más influyentes. Dos de las tres más importantes fueron creadas en el preprocesamiento.
7. **Trabajo futuro:**
   - Incorporar las demás tablas del dataset (historial en buró de crédito, solicitudes y pagos anteriores) mediante agregados por cliente. Esa es la principal fuente de mejora en la competencia.
   - Ensamblar LightGBM con el MLP.
   - Optimizar directamente métricas de negocio (costo de un falso negativo vs un falso positivo).

## Referencias

- Home Credit Group. (2018). *Home Credit Default Risk* [Competencia de Kaggle]. https://www.kaggle.com/competitions/home-credit-default-risk
- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (2.ª ed.). O'Reilly.
- Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR*.
- Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
- Dozat, T. (2016). Incorporating Nesterov momentum into Adam. *ICLR Workshop*.
- Bergstra, J., & Bengio, Y. (2012). Random search for hyper-parameter optimization. *JMLR, 13*, 281–305.
- Bergstra, J., Bardenet, R., Bengio, Y., & Kégl, B. (2011). Algorithms for hyper-parameter optimization. *NeurIPS*.
- Snoek, J., Larochelle, H., & Adams, R. P. (2012). Practical Bayesian optimization of machine learning algorithms. *NeurIPS*.
- Li, L., Jamieson, K., DeSalvo, G., Rostamizadeh, A., & Talwalkar, A. (2018). Hyperband: A novel bandit-based approach to hyperparameter optimization. *JMLR, 18*(185), 1–52.
- Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *KDD*.
- Ke, G., et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. *NeurIPS*.
- O'Malley, T., Bursztein, E., Long, J., Chollet, F., Jin, H., Invernizzi, L., et al. (2019). *KerasTuner*. https://github.com/keras-team/keras-tuner